# 01 — Behavioral Baseline: English vs Turkish Math Reasoning

**Goal**: Establish that Llama-3-8B shows a reasoning gap between English and Turkish on grade school math problems. This motivates the mechanistic analysis in Notebook 02.

**Model**: Meta-Llama-3-8B-Instruct (Instruct variant for behavioral eval)
**Data**: 30 matched problems from GSM8K (EN) and GSM8K-TR (TR)

In [ ]:
# Setup — run this first on Colab
# !pip install -q torch transformers accelerate datasets huggingface-hub tqdm matplotlib seaborn pandas scipy pyyaml python-dotenv einops

import os
import sys
sys.path.insert(0, '..')  # For Colab: adjust to repo root

# Set your HuggingFace token
# os.environ['HF_TOKEN'] = 'your_token_here'

from src.utils import set_seed, setup_logging, get_gpu_info

set_seed(42)
logger = setup_logging()
print(f"GPU: {get_gpu_info()}")

In [ ]:
# Load matched problems
from src.data import load_matched_problems, compute_token_fertility

problems = load_matched_problems(n_problems=30, seed=42)
print(f"Loaded {len(problems)} matched problems")

# Display a sample
p = problems[0]
print(f"\n--- Problem #{p['idx']} ({p['n_steps']} steps, answer={p['answer_number']}) ---")
print(f"EN: {p['question_en'][:200]}...")
print(f"TR: {p['question_tr'][:200]}...")

In [ ]:
# Tokenization analysis
from src.model import LlamaWrapper
import pandas as pd

# Load model (Instruct for behavioral eval)
wrapper = LlamaWrapper(
    model_name="meta-llama/Meta-Llama-3-8B-Instruct",
    dtype="float16",
)

# Compute fertility for all problems
fertility_data = []
for p in problems:
    f = compute_token_fertility(wrapper.tokenizer, p['question_en'], p['question_tr'])
    f['idx'] = p['idx']
    f['answer'] = p['answer_number']
    fertility_data.append(f)

df_fertility = pd.DataFrame(fertility_data)
print("\nTokenization Statistics:")
print(f"  Mean tokens EN: {df_fertility['n_tokens_en'].mean():.1f}")
print(f"  Mean tokens TR: {df_fertility['n_tokens_tr'].mean():.1f}")
print(f"  Mean fertility ratio: {df_fertility['fertility_ratio'].mean():.2f}x")
print(f"  Mean tokens/word EN: {df_fertility['tokens_per_word_en'].mean():.2f}")
print(f"  Mean tokens/word TR: {df_fertility['tokens_per_word_tr'].mean():.2f}")

# Show tokenization example
info = wrapper.get_tokenization_info(problems[0]['question_en'][:100], problems[0]['question_tr'][:100])
print(f"\nExample tokenization:")
print(f"  EN tokens: {info['decoded_en'][:10]}...")
print(f"  TR tokens: {info['decoded_tr'][:10]}...")

In [ ]:
# Run behavioral evaluation
import re
from tqdm import tqdm
from src.data import construct_prompt, extract_answer_number

results = []

for p in tqdm(problems, desc="Evaluating"):
    # English CoT
    prompt_en = construct_prompt(p['question_en'], style='cot_en')
    output_en = wrapper.generate(prompt_en, max_new_tokens=512)
    pred_en = extract_answer_number(output_en)
    
    # Turkish CoT
    prompt_tr = construct_prompt(p['question_tr'], style='cot_tr')
    output_tr = wrapper.generate(prompt_tr, max_new_tokens=512)
    pred_tr = extract_answer_number(output_tr)
    
    results.append({
        'idx': p['idx'],
        'answer': p['answer_number'],
        'n_steps': p['n_steps'],
        'pred_en': pred_en,
        'pred_tr': pred_tr,
        'correct_en': pred_en == p['answer_number'],
        'correct_tr': pred_tr == p['answer_number'],
        'output_en': output_en[:200],
        'output_tr': output_tr[:200],
    })

df_results = pd.DataFrame(results)
print(f"\n=== Behavioral Results ===")
print(f"English accuracy: {df_results['correct_en'].mean() * 100:.1f}%")
print(f"Turkish accuracy: {df_results['correct_tr'].mean() * 100:.1f}%")
print(f"Gap: {(df_results['correct_en'].mean() - df_results['correct_tr'].mean()) * 100:.1f}pp")

In [ ]:
# Accuracy by difficulty
difficulty_bins = {'2-step': (1, 2), '3-4 step': (3, 4), '5-6 step': (5, 6), '7+ step': (7, 100)}
acc_by_difficulty = {}

for label, (lo, hi) in difficulty_bins.items():
    mask = (df_results['n_steps'] >= lo) & (df_results['n_steps'] <= hi)
    subset = df_results[mask]
    if len(subset) > 0:
        acc_by_difficulty[label] = {
            'en': subset['correct_en'].mean() * 100,
            'tr': subset['correct_tr'].mean() * 100,
            'n': len(subset),
        }
        print(f"{label} (n={len(subset)}): EN={acc_by_difficulty[label]['en']:.1f}% TR={acc_by_difficulty[label]['tr']:.1f}%")

In [ ]:
# Error analysis for Turkish failures
tr_failures = df_results[~df_results['correct_tr'] & df_results['correct_en']]
print(f"\nProblems where EN correct but TR wrong: {len(tr_failures)}")
print("\nSample Turkish failure outputs:")
for _, row in tr_failures.head(3).iterrows():
    print(f"  Problem #{row['idx']} (answer={row['answer']}, predicted={row['pred_tr']})")
    print(f"  Output: {row['output_tr'][:150]}...")
    print()

In [ ]:
# Statistical test
from src.metrics import paired_permutation_test
import numpy as np

p_value = paired_permutation_test(
    df_results['correct_en'].values.astype(float),
    df_results['correct_tr'].values.astype(float),
)
print(f"Paired permutation test p-value: {p_value:.4f}")
print(f"Significant at alpha=0.05: {p_value < 0.05}")

In [ ]:
# Save results
from src.visualization import plot_accuracy_comparison
from src.utils import ensure_dir

ensure_dir('../results/figures')
ensure_dir('../results/tables')

# Save accuracy figure
fig = plot_accuracy_comparison(
    acc_en=df_results['correct_en'].mean() * 100,
    acc_tr=df_results['correct_tr'].mean() * 100,
    acc_by_difficulty=acc_by_difficulty,
    save_path='../results/figures/fig1_accuracy_comparison.pdf',
)
fig.savefig('../results/figures/fig1_accuracy_comparison.png', dpi=300)
plt.show()

# Save CSV
df_results.to_csv('../results/tables/behavioral_results.csv', index=False)
df_fertility.to_csv('../results/tables/fertility_analysis.csv', index=False)
print("Results saved.")